In [1]:
from unstructured.partition.pdf import partition_pdf

c:\Users\tchouarr\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
elements = partition_pdf(
    filename="..\data\Guide Utilisateur HM Agenda v1.24.11.pdf",
    strategy="fast",
    extract_images_in_pdf=False,
    languages=["fra"]
)

In [5]:
len(elements)

3039

In [6]:
elements[:5]

## cheking metadata

In [7]:
for el in elements[:50]:
    print(type(el))
    print(el.text)
    print(el.metadata)
    print("=" * 50)

<class 'unstructured.documents.elements.Footer'>
1
<class 'unstructured.documents.elements.Title'>
Guide Utilisateur HM Agenda
<class 'unstructured.documents.elements.Title'>
GUIDE UTILISATEUR HM AGENDA
<class 'unstructured.documents.elements.Title'>
Version 1.2411
<class 'unstructured.documents.elements.Footer'>
GUIDE UTILISATEUR HM AGENDA – COPYRIGHT SOFTWAY MEDICAL – 2024
<class 'unstructured.documents.elements.Title'>
Guide Utilisateur HM Agenda
<class 'unstructured.documents.elements.Title'>
SOMMAIRE
<class 'unstructured.documents.elements.Text'>
1 PREAMBULE .......................................................... 7 Public concerné ........................................................................................................... 7 Prérequis ................................................................................................................... 7 Symboles utilisés .................................................................................................

In [8]:
markdown_text = ""

for el in elements:

    page = getattr(el.metadata, "page_number", None)

    if el.category == "Title":
        markdown_text += f"\n# {el.text}\n"

    else:
        markdown_text += f"\n{el.text}\n"

In [9]:
markdown_text

'\n1\n\n# Guide Utilisateur HM Agenda\n\n# GUIDE UTILISATEUR HM AGENDA\n\n# Version 1.2411\n\nGUIDE UTILISATEUR HM AGENDA – COPYRIGHT SOFTWAY MEDICAL – 2024\n\n# Guide Utilisateur HM Agenda\n\n# SOMMAIRE\n\n1 PREAMBULE .......................................................... 7 Public concerné ........................................................................................................... 7 Prérequis ................................................................................................................... 7 Symboles utilisés .......................................................................................................... 7\n\n2 PRESENTATION DES DIFFERENTS PLANNINGS DE L’AGENDA ................................................................. 8 Planning des rendez-vous ............................................................................................... 9 Planning des ressources (=vacations) ........................................................

In [10]:
elements

 ...]

In [11]:
from langchain_core.documents import Document

docs = []

current_header = "Sans titre"

for el in elements:

    if el.category == "Title":
        current_header = el.text
        continue

    text = el.text.strip()

    if not text:
        continue

    page = getattr(el.metadata, "page_number", None)

    doc = Document(
        page_content=text,
        metadata={
            "header": current_header,
            "page": page,
            "category": el.category,
            "source": el.metadata.filename
        }
    )

    docs.append(doc)

In [12]:
docs

[Document(metadata={'header': 'Sans titre', 'page': 1, 'category': 'Footer', 'source': 'Guide Utilisateur HM Agenda v1.24.11.pdf'}, page_content='1'),
 Document(metadata={'header': 'Version 1.2411', 'page': 1, 'category': 'Footer', 'source': 'Guide Utilisateur HM Agenda v1.24.11.pdf'}, page_content='GUIDE UTILISATEUR HM AGENDA – COPYRIGHT SOFTWAY MEDICAL – 2024'),
 Document(metadata={'header': 'SOMMAIRE', 'page': 2, 'category': 'UncategorizedText', 'source': 'Guide Utilisateur HM Agenda v1.24.11.pdf'}, page_content='1 PREAMBULE .......................................................... 7 Public concerné ........................................................................................................... 7 Prérequis ................................................................................................................... 7 Symboles utilisés .......................................................................................................... 7'),
 Document(metadata={'

In [13]:
len(docs)

1913

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

In [15]:
final_docs = splitter.split_documents(docs)

In [16]:
final_docs

[Document(metadata={'header': 'Sans titre', 'page': 1, 'category': 'Footer', 'source': 'Guide Utilisateur HM Agenda v1.24.11.pdf'}, page_content='1'),
 Document(metadata={'header': 'Version 1.2411', 'page': 1, 'category': 'Footer', 'source': 'Guide Utilisateur HM Agenda v1.24.11.pdf'}, page_content='GUIDE UTILISATEUR HM AGENDA – COPYRIGHT SOFTWAY MEDICAL – 2024'),
 Document(metadata={'header': 'SOMMAIRE', 'page': 2, 'category': 'UncategorizedText', 'source': 'Guide Utilisateur HM Agenda v1.24.11.pdf'}, page_content='1 PREAMBULE .......................................................... 7 Public concerné ........................................................................................................... 7 Prérequis ................................................................................................................... 7 Symboles utilisés .......................................................................................................... 7'),
 Document(metadata={'

In [17]:
len(final_docs)

2018

In [18]:
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [19]:
from dotenv import load_dotenv

In [20]:
# Embedding wrapper for LangChain
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3"
)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 32416.94it/s]


In [21]:
dimension = 1024

In [22]:
persist_directory="embeddings_pdf200p"

In [23]:
chroma_vdb=Chroma.from_documents(documents=final_docs,
                      embedding=embedding_model,
                      persist_directory=persist_directory,
                      )

In [24]:
chroma_vdb.persist()

C:\Users\tchouarr\AppData\Local\Temp\ipykernel_28500\3808622788.py:1: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  chroma_vdb.persist()


### appel de la base

In [25]:
vdb=Chroma(persist_directory=persist_directory,
        embedding_function=embedding_model)

C:\Users\tchouarr\AppData\Local\Temp\ipykernel_28500\35172397.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vdb=Chroma(persist_directory=persist_directory,


In [26]:
retriever=vdb.as_retriever(search_kwargs={"k": 40})

In [27]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [28]:
prompt = ChatPromptTemplate.from_template("""
Tu es un assistant spécialisé dans les documents administratifs français.

Réponds uniquement à partir du contexte fourni.

Contexte:
{context}

Question:
{question}
Si possible de donné le numero de la page,
Si l'information n'est pas présente dans le contexte,
réponds:
"Information non trouvée dans les documents."
""")

In [29]:
chain = prompt | llm | StrOutputParser()

In [41]:
question = "comment calculer le besoin d'un site ?"

docs = retriever.invoke(question)

context = "\n\n".join(
    [doc.page_content for doc in docs]
)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

Le calcul du besoin d'un site peut être effectué en prenant en compte les données de plusieurs sites (ceux appartenant au groupe de site). Pour ce faire, il est possible de déclarer le groupe de site dans les données de tous les sites appartenant au groupe. Les données de consommations, du disponible, des encours fournisseurs, des encours services sont cumulées selon tous les sites du groupe.

Le calcul du besoin net est effectué comme suit :

* Besoin brut = Consommation de la période explorée / nb jours de la période explorée
* Le système prend en compte les paramètres du site enregistrés dans les valeurs paramètres du groupe XAP, tels que le nombre de jours de consommation pour le calcul du besoin net pour les articles ayant un type de délai = 1 ou 2.

Il n'y a pas de numéro de page spécifique pour cette information, car le contexte fourni ne comporte pas de pagination.


In [42]:

question = "comment calculer le besoin ?"

docs = retriever.invoke(question)

context = "\n\n".join(
    [doc.page_content for doc in docs]
)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

Le calcul du besoin net est effectué comme suit :

→ Besoin brut = Consommation de la période explorée / nb jours de la période explorée

Il n'y a pas de numéro de page spécifique fourni dans le contexte, mais cette information se trouve dans la section "5. Calcul du besoin net".

Il est également important de noter que le calcul du besoin net peut varier en fonction du type de délai et des paramètres définis pour chaque article, tels que :

* XNBJRCBNS1 – Nb jr conso sans délai (CBN) pour les articles sans délai
* XNBJRCBNS2 – Nb jr conso délai 1 (CBN) pour les articles ayant un type de délai = 1 ou D1
* XNBJRCBNS3 – Nb jr conso délai 2 (CBN) pour les articles ayant un type de délai = 2 ou D2

Ces paramètres permettent de définir le nombre de jours par défaut pour le calcul du besoin net pour chaque type d'article.


In [30]:
def format_docs(docs):

    formatted = []

    for doc in docs:

        page = doc.metadata.get("page", "Unknown")
        header = doc.metadata.get("header", "Sans titre")
        source = doc.metadata.get("source", "Unknown")

        chunk = f"""
Source: {source}
Page: {page}
Section: {header}

Contenu:
{doc.page_content}
"""

        formatted.append(chunk)

    return "\n\n".join(formatted)

In [ ]:
context = format_docs(retrieved_docs)

In [45]:
question = "comment calculer le besoin ?"

docs = retriever.invoke(question)

context = format_docs(docs)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

Le calcul du besoin est effectué de la manière suivante :

1. Besoin brut = Consommation de la période explorée / nb jours de la période explorée (Page 14)
2. Besoin net = Besoin brut - disponible théorique (Page 15)

Il est également possible de calculer le besoin en fonction du type de délai de l'article :
- Pour les articles sans délai (Type de délai dans la fiche article = blanc), le système prendra les paramètres définis par défaut (Page 9)
- Pour les articles avec un type de délai = 1 ou D1, le système prendra les paramètres définis par XNBJRCBNS2 (Page 5)
- Pour les articles avec un type de délai = 2 ou D2, le système prendra les paramètres définis par XNBJRCBNS3 (Page 5)

Le calcul du disponible théorique est également nécessaire pour calculer le besoin net (Page 14).

Il est important de noter que les paramètres utilisés pour le calcul des besoins peuvent varier en fonction des besoins du gestionnaire et des paramètres définis dans le système.


In [31]:
question = "est-il possible d’avoir une visualisation multi-agents ?"

docs = retriever.invoke(question)

context = format_docs(docs)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

Il est possible d'avoir une visualisation multi-agents, mais cette information n'est pas explicitement mentionnée dans le contexte fourni. Cependant, on peut trouver des informations sur la visualisation de plannings pour plusieurs patients ou intervenants, mais pas spécifiquement sur une visualisation multi-agents.

Cependant, on peut trouver des informations sur la visualisation de plannings pour plusieurs patients ou intervenants, par exemple à la page 41, où il est mentionné que "Si on clique sur le patient ADDA Marc en dessous de « Charger ressource associée », on visualise alors son planning, à côté de celui de notre kinésithérapeute MATHIEU Patrick :".

Mais pour répondre à la question, il n'y a pas d'information explicite sur la visualisation multi-agents.

Réponse:
Information non trouvée dans les documents.
